# 課題5：映画レビューの評判分析

本課題ではAmazon傘下の「IMDb」に投稿された映画のレビュー（英語）を分析し、レビューがPositive（ポジティブ）か、Negative（ネガティブ）かの判別を行ないます。

データセットは、以下のサイトで配布されているものを利用します。

[Large Movie Review Dataset](https://ai.stanford.edu/%7Eamaas/data/sentiment/)

わからない場合は、ここまでのレッスン内容や各種ライブラリの公式ドキュメントを参照しましょう。

## 1. 必要なライブラリのimport

In [2]:
# （変更しないでください）

# 必要なライブラリのimport
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 文章ファイル検索用
import glob
import collections
from sklearn.feature_extraction import DictVectorizer

# DataFrameですべての列を表示する設定
pd.options.display.max_columns = None

# seabornによる装飾を適用する
sns.set_theme()

## 2. データの読み込み

In [2]:
# ダウンロードした圧縮ファイルを解凍する（変更しないでください）
!tar zxvf aclImdb_v1.tar.gz

x aclImdb/
x aclImdb/test/
x aclImdb/train/
x aclImdb/test/neg/
x aclImdb/test/pos/
x aclImdb/train/neg/
x aclImdb/train/pos/
x aclImdb/train/unsup/
x aclImdb/imdbEr.txt
x aclImdb/imdb.vocab
x aclImdb/README
x aclImdb/test/labeledBow.feat
x aclImdb/test/urls_neg.txt
x aclImdb/test/urls_pos.txt
x aclImdb/train/unsupBow.feat
x aclImdb/train/labeledBow.feat
x aclImdb/train/urls_neg.txt
x aclImdb/train/urls_pos.txt
x aclImdb/train/urls_unsup.txt
x aclImdb/test/neg/127_3.txt
x aclImdb/test/neg/126_4.txt
x aclImdb/test/neg/125_3.txt
x aclImdb/test/neg/124_2.txt
x aclImdb/test/neg/123_4.txt
x aclImdb/test/neg/122_4.txt
x aclImdb/test/neg/121_4.txt
x aclImdb/test/neg/120_2.txt
x aclImdb/test/neg/119_3.txt
x aclImdb/test/neg/118_1.txt
x aclImdb/test/neg/117_1.txt
x aclImdb/test/neg/116_4.txt
x aclImdb/test/neg/115_3.txt
x aclImdb/test/neg/114_2.txt
x aclImdb/test/neg/113_3.txt
x aclImdb/test/neg/112_2.txt
x aclImdb/test/neg/111_3.txt
x aclImdb/test/neg/110_1.txt
x aclImdb/test/neg/109_4.txt
x a

*./aclImdb* フォルダ内にあるファイルを読み込みます。

In [3]:
# trainフォルダのファイル一覧を取得（変更しないでください）
train_neg_files = glob.glob("./aclImdb/train/neg/*")
train_pos_files = glob.glob("./aclImdb/train/pos/*")

# testフォルダのファイル一覧を取得（変更しないでください）
test_neg_files = glob.glob("./aclImdb/test/neg/*")
test_pos_files = glob.glob("./aclImdb/test/pos/*")

In [4]:
# それぞれのファイル数を確認
print(len(train_neg_files))
print(len(train_pos_files))
print(len(test_neg_files))
print(len(test_pos_files))

12500
12500
12500
12500


前処理をするため、合計50000あるファイルをリストにまとめます。

In [5]:
# ファイル名をまとめたリストを用意（変更しないでください）
filenames = train_neg_files + train_pos_files + test_neg_files + test_pos_files

# filenamesの長さを確認（変更しないでください）
len(filenames)

50000

リストの最初と最後のファイルを確認してみます。

In [6]:
# エンコーディング用定数（変更しないでください）
ENCODING = 'utf-8'

In [7]:
# 最初のファイルの内容を確認
with open(filenames[0], 'r', encoding=ENCODING) as f:
    text = f.read()
    print(text)

Story of a man who has unnatural feelings for a pig. Starts out with a opening scene that is a terrific example of absurd comedy. A formal orchestra audience is turned into an insane, violent mob by the crazy chantings of it's singers. Unfortunately it stays absurd the WHOLE time with no general narrative eventually making it just too off putting. Even those from the era should be turned off. The cryptic dialogue would make Shakespeare seem easy to a third grader. On a technical level it's better than you might think with some good cinematography by future great Vilmos Zsigmond. Future stars Sally Kirkland and Frederic Forrest can be seen briefly.


In [8]:
# 最後のファイルの内容を確認
with open(filenames[-1], 'r', encoding=ENCODING) as f:
    text = f.read()
    print(text)

I've seen this story before but my kids haven't. Boy with troubled past joins military, faces his past, falls in love and becomes a man. The mentor this time is played perfectly by Kevin Costner; An ordinary man with common everyday problems who lives an extraordinary conviction, to save lives. After losing his team he takes a teaching position training the next generation of heroes. The young troubled recruit is played by Kutcher. While his scenes with the local love interest are a tad stiff and don't generate enough heat to melt butter, he compliments Costner well. I never really understood Sela Ward as the neglected wife and felt she should of wanted Costner to quit out of concern for his safety as opposed to her selfish needs. But her presence on screen is a pleasure. The two unaccredited stars of this movie are the Coast Guard and the Sea. Both powerful forces which should not be taken for granted in real life or this movie. The movie has some slow spots and could have used the wa

## 3. データの前処理

データの前処理として、形態素解析と行列への変換を行ないます。

### 形態素解析

In [9]:
# 文字列の中で使われている単語ごとの数を返す関数を作成
#（レッスン本編の内容を確認して、下記にコードを追記してください）
def get_word_count(text, min_length=3):
    # ノイズの除去：不要と思われる文字を除去する
    for ch in ".,:;!?-+*/=()[]{}<>~^#$@%&'\"_0123456789":
        text = text.replace(ch, ' ')

    # 形態素解析：文章を単語に分割
    _words = text.strip().split()

    # 表記のゆれの補正：
    # 単語のリストを受け取り、指定された文字数以上の単語だけをすべて小文字にして返す
    _words = [_word.lower() for _word in _words if len(_word) >= min_length]

    # collections.Counterの戻り値は辞書型のサブクラス
    _count = collections.Counter(_words)

    # 辞書型に変換して返す
    return dict(_count)

In [10]:
# 最初のファイルを使って、先ほど作成した関数をテスト
with open(filenames[0], 'r', encoding=ENCODING) as f:
    text = f.read()

get_word_count(text)

{'story': 1,
 'man': 1,
 'who': 1,
 'has': 1,
 'unnatural': 1,
 'feelings': 1,
 'for': 1,
 'pig': 1,
 'starts': 1,
 'out': 1,
 'with': 3,
 'opening': 1,
 'scene': 1,
 'that': 1,
 'terrific': 1,
 'example': 1,
 'absurd': 2,
 'comedy': 1,
 'formal': 1,
 'orchestra': 1,
 'audience': 1,
 'turned': 2,
 'into': 1,
 'insane': 1,
 'violent': 1,
 'mob': 1,
 'the': 4,
 'crazy': 1,
 'chantings': 1,
 'singers': 1,
 'unfortunately': 1,
 'stays': 1,
 'whole': 1,
 'time': 1,
 'general': 1,
 'narrative': 1,
 'eventually': 1,
 'making': 1,
 'just': 1,
 'too': 1,
 'off': 2,
 'putting': 1,
 'even': 1,
 'those': 1,
 'from': 1,
 'era': 1,
 'should': 1,
 'cryptic': 1,
 'dialogue': 1,
 'would': 1,
 'make': 1,
 'shakespeare': 1,
 'seem': 1,
 'easy': 1,
 'third': 1,
 'grader': 1,
 'technical': 1,
 'level': 1,
 'better': 1,
 'than': 1,
 'you': 1,
 'might': 1,
 'think': 1,
 'some': 1,
 'good': 1,
 'cinematography': 1,
 'future': 2,
 'great': 1,
 'vilmos': 1,
 'zsigmond': 1,
 'stars': 1,
 'sally': 1,
 'kirkland':

In [11]:
# 単語ごとの数のリストを作成（変更しないでください）
word_count_data = []

In [12]:
# すべてのファイルに対して、先ほど作成した関数を実行
for filename in filenames:
    with open(filename, 'r', encoding=ENCODING) as f:
        text = f.read()
        count = get_word_count(text)
        word_count_data.append(count)

In [25]:
# 単語ごとの数のリストの長さを確認
word_count_data

[{'story': 1,
  'man': 1,
  'who': 1,
  'has': 1,
  'unnatural': 1,
  'feelings': 1,
  'for': 1,
  'pig': 1,
  'starts': 1,
  'out': 1,
  'with': 3,
  'opening': 1,
  'scene': 1,
  'that': 1,
  'terrific': 1,
  'example': 1,
  'absurd': 2,
  'comedy': 1,
  'formal': 1,
  'orchestra': 1,
  'audience': 1,
  'turned': 2,
  'into': 1,
  'insane': 1,
  'violent': 1,
  'mob': 1,
  'the': 4,
  'crazy': 1,
  'chantings': 1,
  'singers': 1,
  'unfortunately': 1,
  'stays': 1,
  'whole': 1,
  'time': 1,
  'general': 1,
  'narrative': 1,
  'eventually': 1,
  'making': 1,
  'just': 1,
  'too': 1,
  'off': 2,
  'putting': 1,
  'even': 1,
  'those': 1,
  'from': 1,
  'era': 1,
  'should': 1,
  'cryptic': 1,
  'dialogue': 1,
  'would': 1,
  'make': 1,
  'shakespeare': 1,
  'seem': 1,
  'easy': 1,
  'third': 1,
  'grader': 1,
  'technical': 1,
  'level': 1,
  'better': 1,
  'than': 1,
  'you': 1,
  'might': 1,
  'think': 1,
  'some': 1,
  'good': 1,
  'cinematography': 1,
  'future': 2,
  'great': 1,


In [27]:
# 単語ごとの数のリストの0番目を表示
word_count_data[0]

{'story': 1,
 'man': 1,
 'who': 1,
 'has': 1,
 'unnatural': 1,
 'feelings': 1,
 'for': 1,
 'pig': 1,
 'starts': 1,
 'out': 1,
 'with': 3,
 'opening': 1,
 'scene': 1,
 'that': 1,
 'terrific': 1,
 'example': 1,
 'absurd': 2,
 'comedy': 1,
 'formal': 1,
 'orchestra': 1,
 'audience': 1,
 'turned': 2,
 'into': 1,
 'insane': 1,
 'violent': 1,
 'mob': 1,
 'the': 4,
 'crazy': 1,
 'chantings': 1,
 'singers': 1,
 'unfortunately': 1,
 'stays': 1,
 'whole': 1,
 'time': 1,
 'general': 1,
 'narrative': 1,
 'eventually': 1,
 'making': 1,
 'just': 1,
 'too': 1,
 'off': 2,
 'putting': 1,
 'even': 1,
 'those': 1,
 'from': 1,
 'era': 1,
 'should': 1,
 'cryptic': 1,
 'dialogue': 1,
 'would': 1,
 'make': 1,
 'shakespeare': 1,
 'seem': 1,
 'easy': 1,
 'third': 1,
 'grader': 1,
 'technical': 1,
 'level': 1,
 'better': 1,
 'than': 1,
 'you': 1,
 'might': 1,
 'think': 1,
 'some': 1,
 'good': 1,
 'cinematography': 1,
 'future': 2,
 'great': 1,
 'vilmos': 1,
 'zsigmond': 1,
 'stars': 1,
 'sally': 1,
 'kirkland':

### 行列への変換

In [28]:
# DictVectorizerを使用して行列に変換し、datasetに格納する
vec = DictVectorizer()
dataset = vec.fit_transform(word_count_data)

In [29]:
# datasetの大きさを確認
dataset

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6117713 stored elements and shape (50000, 101249)>

In [30]:
# 各列に対応した単語を取得
vec.get_feature_names_out()

array(['\x08\x08\x08\x08a', '\x10own', '\\and\\', ..., '…although',
       '…but', '…until'], dtype=object)

## 4. 機械学習の実施

In [31]:
# 必要なライブラリの追加import（変更しないでください）
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

目的変数と説明変数を用意します。

In [33]:
# 目的変数Yの用意
# neg12500 + pos12500 + neg12500 + pos12500 = 50000
Y = [0]*12500 + [1]*12500 + [0]*12500 + [1]*12500

In [41]:
# 上記のY、および前処理されたdatasetからデータを分割し、
# X_train, Y_train, X_test, Y_testに格納する
#
# 詳細：
#   - dataset の先頭から25000件を 変数 X_train に、残りを変数 X_test に代入
#   - 目的変数 Y の先頭から25000件を 変数 Y_train に、残りを Y_test に代入
X_train = dataset[:25000]
X_test = dataset[25000:]
Y_train = Y[:25000]
Y_test = Y[25000:]


In [42]:
# X_trainとY_trainを、train_test_splitで7:3に分割し、3割のほうを検証データ（X_valid, Y_valid）にする
X_train, X_valid, Y_train, Y_valid = train_test_split(X_train, Y_train, test_size=0.3, random_state=0)

In [47]:
# ロジスティック回帰モデルを作成し、学習して、検証データによる予測を実施する
logistic_model = LogisticRegression(max_iter=2000)
logistic_model.fit(X_train, Y_train)
Y_pred = logistic_model.predict(X_valid)

# classification_reportを実行し、検証データによるモデルの評価を行なう
print(classification_report(Y_valid, Y_pred))

              precision    recall  f1-score   support

           0       0.88      0.87      0.88      3767
           1       0.87      0.88      0.88      3733

    accuracy                           0.88      7500
   macro avg       0.88      0.88      0.88      7500
weighted avg       0.88      0.88      0.88      7500



## 5. テストデータによる評価

最後に、テストデータで評価を行ないましょう。

In [50]:
# テストデータで予測を実施する
Y_pred = logistic_model.predict(X_test)

# classification_reportを実行し、テストデータによるモデルの評価を行なう
print(classification_report(Y_test, Y_pred))

              precision    recall  f1-score   support

           0       0.86      0.87      0.86     12500
           1       0.86      0.86      0.86     12500

    accuracy                           0.86     25000
   macro avg       0.86      0.86      0.86     25000
weighted avg       0.86      0.86      0.86     25000

